# 04 – Task 3 (KV cache) and Task 4 (tiled attention): correctness + benchmarks
Uses the base `Qwen2.5-1.5B-Instruct` (fp16, T4). Code: `src/manual_decoder.py`, `src/kv_decode.py`, `src/attention.py`.
Takes about 10–15 min in total.

In [ ]:
# ==== EDIT THESE TWO LINES ====
GITHUB_USER = "FatimaRana50"
GIT_NAME    = "Fatima Rana"
# ==============================
REPO = "trialmatch"

!pip -q install peft datasets
!pip -q uninstall -y torchao       # Colab ships torchao 0.10; peft refuses to load with it (we do not use it)
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
token = userdata.get('GH_TOKEN')          # Colab sidebar -> key icon -> add secret GH_TOKEN
if not os.path.exists(f'/content/{REPO}'):
    !git clone -q https://{token}@github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git pull -q
!git config user.name "{GIT_NAME}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
import sys; sys.path.insert(0, '/content/trialmatch/src')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Task 4: reference vs tiled online-softmax attention

In [ ]:
!python src/bench_attention.py

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
c = json.load(open("results/attention_correctness.json"))
print(f"correctness: {c['passed']}/{c['total']} passed")
print(pd.DataFrame(c["cases"]).to_string())
print(pd.DataFrame(json.load(open("results/attention_bench.json"))["rows"]).to_string())
display(Image("results/attention_bench.png"))

## Task 3: KV-cached vs uncached greedy decoding

In [ ]:
!python src/bench_kv.py

In [ ]:
c = json.load(open("results/kv_correctness.json"))
print(f"token-equality: {c['passed']}/{c['total']} passed"); print(c["env"])
print(pd.DataFrame(c["checks"]).to_string())
b = pd.DataFrame(json.load(open("results/kv_bench.json"))["rows"])
print(b.round(4).to_string())
display(Image("results/kv_bench.png"))

In [ ]:
!pip freeze > logs/colab_env_04.txt
!git add results/ logs/ && git commit -m "KV-cache and attention benchmarks (T4)" && git pull -q --rebase && git push